# Final Project: Diabetes Risk Screening
## Threshold Analysis and Reliability of the Risk Scores

**Name:** TORRECAMPO  
**Course:** CPE ELEC 1 - CpE Elective 1

This notebook answers two questions about the saved model:
1. Which classification threshold should be used for screening?
2. How reliable are the predicted scores, and how should they be interpreted?

Run notebook `01_data_preparation_and_model_training.ipynb` first. This notebook uses the model that it saved.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, brier_score_loss
from sklearn.calibration import calibration_curve

PROJECT = '/content/drive/MyDrive/CPE ELEC 1 TORRECAMPO/FINAL PROJECT TORRECAMPO/'
CSV_FILE = PROJECT + 'data/diabetes_binary_health_indicators_BRFSS2015.csv'
MODEL_FILE = PROJECT + 'model/diabetes_model.joblib'
INFO_FILE = PROJECT + 'model/model_info.json'

## 2. Load Data, Model and the Same Test Set

The data preparation and the split are repeated exactly as in notebook 01 (same features, same duplicate removal, `random_state=42`), so the test set is the same one used to evaluate the model.

In [ ]:
with open(INFO_FILE, 'r') as f:
    model_info = json.load(f)
FEATURES = model_info['features']

df = pd.read_csv(CSV_FILE)
df = df.drop_duplicates()
X = df[FEATURES]
y = df['Diabetes_binary'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

model = joblib.load(MODEL_FILE)
y_prob = model.predict_proba(X_test)[:, 1]
print('Model:', model_info['selected_model'])
print('Test records:', len(y_test))
print('Diabetes share in test set:', round(y_test.mean() * 100, 2), '%')

## 3. Threshold Analysis

The model gives every person a score from 0 to 1. A person is shown as Higher risk when the score is at or above the threshold. A lower threshold finds more people with diabetes (higher recall) but also flags more people without diabetes (lower precision).

In [ ]:
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
rows = []
for t in thresholds:
    pred = (y_prob >= t).astype(int)
    rows.append({
        'Threshold': t,
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'Flagged (%)': pred.mean() * 100,
    })
threshold_table = pd.DataFrame(rows).round(3)
threshold_table

In [ ]:
plt.plot(threshold_table['Threshold'], threshold_table['Precision'], marker='o', label='Precision')
plt.plot(threshold_table['Threshold'], threshold_table['Recall'], marker='o', label='Recall')
plt.plot(threshold_table['Threshold'], threshold_table['F1'], marker='o', label='F1')
plt.axvline(0.5, linestyle='--', color='gray')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall at Different Thresholds')
plt.legend()
plt.savefig(PROJECT + 'results/threshold_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

**Choice of threshold:** For screening, missing a person who has diabetes is worse than an extra blood test, so recall matters more than precision. The threshold of 0.5 is kept because it finds most of the people who reported diabetes while keeping the number of flagged people reasonable. A lower threshold would find more cases but would flag many more people who do not have diabetes. Read the table above to see this trade-off.

## 4. Reliability of the Predicted Scores

The model was trained with balanced classes, so its scores are not real probabilities. The Brier score and the calibration curve check this. A perfectly reliable score would lie on the diagonal line (a score of 0.3 means 30% of such people have diabetes).

In [ ]:
brier = brier_score_loss(y_test, y_prob)
brier_baseline = brier_score_loss(y_test, np.repeat(y_train.mean(), len(y_test)))
print('Brier score of the model scores:', round(brier, 4))
print('Brier score of always guessing the average:', round(brier_baseline, 4))

In [ ]:
observed, predicted = calibration_curve(y_test, y_prob, n_bins=10, strategy='quantile')
plt.plot(predicted, observed, marker='o', label='Model scores')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly reliable')
plt.xlabel('Average model score')
plt.ylabel('Share who reported diabetes')
plt.title('Calibration Curve')
plt.legend()
plt.savefig(PROJECT + 'results/calibration_curve.png', dpi=120, bbox_inches='tight')
plt.show()

**Reading the curve:** Points below the diagonal mean the score is higher than the real share of people who reported diabetes. The scores are therefore a relative risk indicator, not a probability. To interpret a score, the application uses the score ranges below, which show what actually happened in the test data.

## 5. Score Ranges and Observed Rates

In [ ]:
edges = [0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0001]
band_rows = []
for i in range(len(edges) - 1):
    low = edges[i]
    high = edges[i + 1]
    mask = (y_prob >= low) & (y_prob < high)
    count = int(mask.sum())
    if count > 0:
        rate = round(float(y_test.values[mask].mean() * 100), 1)
        label = str(int(low * 100)) + '% to ' + str(min(int(round(high * 100)), 100)) + '%'
        band_rows.append({'low': low, 'high': high, 'label': label, 'count': count, 'diabetes_rate': rate})
bands = pd.DataFrame(band_rows)
bands[['label', 'count', 'diabetes_rate']]

## 6. Save Results for the Application

In [ ]:
model_info['threshold'] = 0.5
model_info['brier_score'] = round(float(brier), 4)
model_info['brier_score_baseline'] = round(float(brier_baseline), 4)
model_info['threshold_analysis'] = threshold_table.to_dict(orient='records')
model_info['score_bands'] = band_rows

with open(INFO_FILE, 'w') as f:
    json.dump(model_info, f, indent=2)

threshold_table.to_csv(PROJECT + 'results/threshold_analysis.csv', index=False)
bands[['label', 'count', 'diabetes_rate']].to_csv(PROJECT + 'results/score_ranges.csv', index=False)
print('Saved. The application will now show the threshold explanation and score ranges.')